In [1]:
import os
import gc
import math
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, TensorDataset
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


In [2]:
SAVE_PATH = "/kaggle/working/checkpoints/"

os.makedirs(SAVE_PATH, exist_ok=True)

In [3]:
import os

for root, dirs, files in os.walk("/kaggle/input"):
    for f in files:
        if "LOOPSEA_10" in f:
            print(os.path.join(root, f))

In [4]:
import os

for root, dirs, files in os.walk("/kaggle/working"):
    for f in files:
        print(os.path.join(root, f))

/kaggle/working/__notebook__.ipynb


In [5]:
files = {
    10: "/kaggle/input/datasets/chetannarware01/pims-bay/missing_data/pemsbay_10percent_missing.csv",
    20: "/kaggle/input/datasets/chetannarware01/pims-bay/missing_data/pemsbay_20percent_missing.csv",
    40: "/kaggle/input/datasets/chetannarware01/pims-bay/missing_data/pemsbay_40percent_missing.csv",
    80: "/kaggle/input/datasets/chetannarware01/pims-bay/missing_data/pemsbay_80percent_missing.csv",
     0: "/kaggle/input/datasets/chetannarware01/pims-bay/missing_data/pemsbay_0percent_missing.csv"
}
print("Files configured:", list(files.keys()))

Files configured: [10, 20, 40, 80, 0]


In [6]:
def create_sequences(data, mask, seq_len=10):
    X, M, Y = [], [], []
    for i in range(len(data) - seq_len):
        X.append(data[i:i+seq_len])
        M.append(mask[i:i+seq_len])
        Y.append(data[i+seq_len])
    return np.array(X), np.array(M), np.array(Y)

In [7]:
class TrafficDataset(Dataset):
    def __init__(self, X, M, Y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.M = torch.tensor(M, dtype=torch.float32)
        self.Y = torch.tensor(Y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.M[idx], self.Y[idx]

In [8]:
class ConvLSTMCell(nn.Module):
    def __init__(self, input_dim, hidden_dim):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.conv = nn.Conv1d(
            input_dim + hidden_dim,
            4 * hidden_dim,
            kernel_size=3,
            padding=1
        )
        self.dropout = nn.Dropout(0.1)

    def forward(self, x, h, c):
        combined = torch.cat([x, h], dim=1)          # (B, input+hidden, F)
        i, f, o, g = torch.chunk(self.conv(combined), 4, dim=1)
        i = torch.sigmoid(i)
        f = torch.sigmoid(f)
        o = torch.sigmoid(o)
        g = torch.tanh(g)
        c = f * c + i * g
        h = o * torch.tanh(c)
        h = self.dropout(h)
        return h, c

In [9]:
class IConvLSTM(nn.Module):
    """
    ConvLSTM with imputation unit following Cui et al. (2020).

    Key fixes vs previous version:
      - Imputation uses BOTH h and c (cell state), matching Eq.10 of paper
      - Returns only the LAST timestep output (not mean over all T)
      - No broken residual addition across mismatched channel dims
      - Mask is fed into all gates (matching LSTM-I Eqs. 13-16)
    """
    def __init__(self, input_dim, hidden_dim=64):
        super().__init__()
        self.hidden_dim = hidden_dim

        # Imputation unit: infers x̃_t from preceding h and c (Eq. 10)
        # Concat h2 + c2 along channel dim → 2*hidden_dim channels
        self.impute = nn.Sequential(
            nn.Conv1d(2 * hidden_dim, hidden_dim, kernel_size=1),
            nn.ReLU(),
            nn.Conv1d(hidden_dim, 1, kernel_size=1)
            # no activation — allows full N(0,1) range matching normalized data
        )

        # Layer 1: input is 1 (single-channel speed) + 1 (mask channel)
        # We concatenate mask as an extra input channel (matching Eqs. 13-16)
        self.convlstm1 = ConvLSTMCell(input_dim=2, hidden_dim=hidden_dim)

        # Layer 2: input is hidden_dim from layer 1
        self.convlstm2 = ConvLSTMCell(input_dim=hidden_dim, hidden_dim=hidden_dim)

        # Final 1x1 conv to map hidden → prediction
        self.output_head = nn.Conv1d(hidden_dim, 1, kernel_size=1)

    def forward(self, x, mask):
        """
        x    : (B, T, F)   normalized speed sequences
        mask : (B, T, F)   1=observed, 0=missing
        Returns:
          pred        : (B, F)   prediction for timestep T+1
          imputations : (B, T, F) inferred values at each step
        """
        B, T, F = x.shape

        # Initialize states
        h1 = torch.zeros(B, self.hidden_dim, F, device=x.device)
        c1 = torch.zeros(B, self.hidden_dim, F, device=x.device)
        h2 = torch.zeros(B, self.hidden_dim, F, device=x.device)
        c2 = torch.zeros(B, self.hidden_dim, F, device=x.device)

        imputations = []

        for t in range(T):
            xt = x[:, t, :].unsqueeze(1)       # (B, 1, F)
            mt = mask[:, t, :].unsqueeze(1)     # (B, 1, F)

            # --- Imputation unit (Eq. 10) ---
            # Infer missing values from previous layer-2 h AND c
            x_tilde = self.impute(torch.cat([h2, c2], dim=1))   # (B, 1, F)

            # --- Mask fusion (Eq. 11-12) ---
            # Use observed value where available, imputed value where missing
            x_prime = mt * xt + (1.0 - mt) * x_tilde            # (B, 1, F)

            imputations.append(x_tilde.squeeze(1))               # (B, F)

            # --- Layer 1: feed x_prime + mask as 2-channel input ---
            x_with_mask = torch.cat([x_prime, mt], dim=1)        # (B, 2, F)
            h1, c1 = self.convlstm1(x_with_mask, h1, c1)

            # --- Layer 2 ---
            h2, c2 = self.convlstm2(h1, h2, c2)

        # Predict only from LAST hidden state (matches paper Eq. 2)
        pred = self.output_head(h2).squeeze(1)                   # (B, F)
        imputations = torch.stack(imputations, dim=1)            # (B, T, F)

        return pred, imputations

In [10]:
def compute_loss(pred, target, imputations, x_input, mask, lam=0.1):
    """
    Combined prediction + imputation regularization loss.
    Matches Eq. 22 of the paper:
      L = Loss(pred, target) + λ * Σ |x_t - x̃_t| over OBSERVED positions

    Critical: imputation error is measured on OBSERVED values (mask==1)
    because those are the only positions where we have ground truth.
    """
    # Prediction loss: MAE + small MSE term for sharper gradients
    pred_loss = torch.mean(torch.abs(pred - target)) + \
                0.1 * torch.mean((pred - target) ** 2)

    # Imputation regularization on observed positions only
    # mask shape: (B, T, F), imputations shape: (B, T, F), x_input shape: (B, T, F)
    imp_error = torch.abs(imputations - x_input) * mask   # zero out missing positions
    imp_loss  = imp_error.sum() / (mask.sum() + 1e-8)

    return pred_loss + lam * imp_loss

In [11]:
def hybrid_loss(pred, target):
    mse = torch.mean((pred - target) ** 2)
    mae = torch.mean(torch.abs(pred - target))
    return 0.5 * mse + mae

In [12]:
from tqdm.auto import tqdm
import gc

def train_model(model, train_loader, val_loader, percent,
                max_epochs=80, patience=15):
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=5, min_lr=1e-5
    )
    
    best_loss = float("inf")
    wait = 0
    best_path = f"/kaggle/working/best_{percent}.pt"
    
    for epoch in range(max_epochs):
        model.train()
        train_loss = 0.0
        
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{max_epochs}", leave=False)
        for x, m, y in pbar:
            # BUG FIX 1: with DataParallel always use 'device' not curr_device
            # DataParallel handles distribution internally from the base device
            x, m, y = x.to(device), m.to(device), y.to(device)
            
            optimizer.zero_grad()
            pred, imputations = model(x, m)
            loss = compute_loss(pred, y, imputations, x, m, lam=0.1)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            
            train_loss += loss.item()
            pbar.set_postfix(loss=f"{loss.item():.4f}")
            
        train_loss /= len(train_loader)
        
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for x, m, y in val_loader:
                # BUG FIX 1: same fix here
                x, m, y = x.to(device), m.to(device), y.to(device)
                pred, _ = model(x, m)
                val_loss += (torch.mean(torch.abs(pred - y)) +
                             0.1 * torch.mean((pred - y) ** 2)).item()
        val_loss /= len(val_loader)
        
        current_lr = optimizer.param_groups[0]['lr']  # BUG FIX 2: [0]['lr'] not ['lr']
        scheduler.step(val_loss)
        new_lr = optimizer.param_groups[0]['lr']
        lr_msg = f"  → LR dropped to {new_lr:.2e}" if new_lr < current_lr else ""
        
        print(f"Epoch {epoch+1:02d} | Train: {train_loss:.4f} | Val: {val_loss:.4f} | "
              f"LR: {current_lr:.2e}{lr_msg}")
        
        if val_loss < best_loss:
            best_loss = val_loss
            wait = 0
            state = model.module.state_dict() if isinstance(model, nn.DataParallel) \
                    else model.state_dict()
            torch.save({"model": state}, best_path)  # BUG FIX 3: mean/std saved in
            print(f"  ✓ Saved checkpoint to {best_path}")   # train_single_dataset not here
        else:
            wait += 1
            if wait >= patience:
                print("  Early stopping triggered")
                break
        
        gc.collect()
        torch.cuda.empty_cache()
    
    # Load best weights back before returning
    state = torch.load(best_path)["model"]
    if isinstance(model, nn.DataParallel):
        model.module.load_state_dict(state)
    else:
        model.load_state_dict(state)
    
    return model

In [13]:
def evaluate_model(model, loader, mean, std):
    model.eval()
    preds, trues, masks = [], [], []

    with torch.no_grad():
        for x, m, y in loader:
            x, m = x.to(device), m.to(device)
            pred, _ = model(x, m)
            preds.append(pred.cpu().numpy())
            trues.append(y.numpy())
            masks.append(m[:, -1, :].cpu().numpy())

    preds = np.concatenate(preds)   # (N, F)
    trues = np.concatenate(trues)   # (N, F)
    masks = np.concatenate(masks)   # (N, F)

    # --- Normalized metrics ---
    mae_norm  = mean_absolute_error(trues.ravel(), preds.ravel())
    rmse_norm = np.sqrt(mean_squared_error(trues.ravel(), preds.ravel()))

    # --- Denormalize: mean/std shape (1, F) → broadcasts correctly ---
    preds_real = preds * std + mean   # (N, F)
    trues_real = trues * std + mean   # (N, F)

    mae_real  = mean_absolute_error(trues_real.ravel(), preds_real.ravel())
    rmse_real = np.sqrt(mean_squared_error(trues_real.ravel(), preds_real.ravel()))

    # Per-sensor R2, then average
    r2_list = [r2_score(trues_real[:, i], preds_real[:, i])
               for i in range(trues_real.shape[1])]
    r2_real = np.mean(r2_list)

    # MAPE on observed, non-zero ground truth
    valid = (np.abs(trues_real) > 1e-3) & (masks == 1)
    mape  = np.mean(np.abs((trues_real[valid] - preds_real[valid]) /
                            trues_real[valid])) * 100

    print("\n==============================")
    print("Normalized Results")
    print("==============================")
    print(f"MAE  : {mae_norm:.4f}")
    print(f"RMSE : {rmse_norm:.4f}")
    print("\n==============================")
    print("Denormalized Results (Real Scale)")
    print("==============================")
    print(f"MAE  : {mae_real:.4f}")
    print(f"RMSE : {rmse_real:.4f}")
    print(f"R2   : {r2_real:.4f}")
    print(f"MAPE : {mape:.2f}%")

    return mae_real, rmse_real, r2_real, mape

In [14]:
def train_single_dataset(percent, model_name="pemabay_iconv"):
    print(f"\n{'='*30}\nSTARTING EXPERIMENT: {percent}% MISSING\n{'='*30}")
    
    raw = pd.read_csv(files[percent])
    raw = raw.apply(pd.to_numeric, errors='coerce')
    
    mask = (~raw.isna()).astype(np.float32).values
    data = raw.values.astype(np.float32)
    
    # Warm-start logic
    mask[:200, :] = 1.0
    col_medians = np.nanmedian(data, axis=0)
    col_medians = np.where(np.isnan(col_medians), 0.0, col_medians)
    
    for col in range(data.shape[1]):
        nan_rows = np.isnan(data[:200, col])
        if nan_rows.any():
            data[:200, col][nan_rows] = col_medians[col]
    
    split = int(0.8 * len(data))
    train_data, val_data = data[:split],  data[split:]
    train_mask, val_mask = mask[:split],  mask[split:]
    
    obs_train = train_data.copy()
    obs_train[train_mask == 0] = np.nan
    mean = np.nanmean(obs_train, axis=0, keepdims=True)
    std  = np.nanstd(obs_train,  axis=0, keepdims=True)
    mean = np.where(np.isnan(mean), 0.0, mean)
    std  = np.where(np.isnan(std) | (std < 1e-8), 1.0, std)
    
    print(f"--- Statistics for {percent}% Missing ---")
    print(f"Mean (first 5 sensors): {mean[0, :5]}")
    print(f"Std  (first 5 sensors): {std[0, :5]}")
    
    train_norm = np.where(train_mask == 1, (train_data - mean) / std, 0.0)
    val_norm   = np.where(val_mask   == 1, (val_data   - mean) / std, 0.0)
    
    SEQ_LEN = 10
    X_tr, M_tr, Y_tr = create_sequences(train_norm, train_mask, SEQ_LEN)
    X_vl, M_vl, Y_vl = create_sequences(val_norm,   val_mask,   SEQ_LEN)
    
    train_loader = DataLoader(
        TensorDataset(
            torch.tensor(X_tr, dtype=torch.float32),
            torch.tensor(M_tr, dtype=torch.float32),
            torch.tensor(Y_tr, dtype=torch.float32)
        ),
        batch_size=64,
        shuffle=True,
        num_workers=2,
        pin_memory=True,
        persistent_workers=True,
        prefetch_factor=2,
        multiprocessing_context='fork'
    )
    val_loader = DataLoader(
        TensorDataset(
            torch.tensor(X_vl, dtype=torch.float32),
            torch.tensor(M_vl, dtype=torch.float32),
            torch.tensor(Y_vl, dtype=torch.float32)
        ),
        batch_size=64,
        shuffle=False,
        num_workers=2,
        pin_memory=True,
        persistent_workers=True,
        prefetch_factor=2,
        multiprocessing_context='fork'
    )
    
    input_dim = X_tr.shape[2]
    model = IConvLSTM(input_dim=input_dim, hidden_dim=64).to(device)
    
    if torch.cuda.device_count() > 1:
        print(f"Using {torch.cuda.device_count()} GPUs with DataParallel")
        model = nn.DataParallel(model)
    else:
        print(f"Using single device: {device}")
    
    print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")
    
    model = train_model(model, train_loader, val_loader, percent)
    
    state = model.module.state_dict() if isinstance(model, nn.DataParallel) \
            else model.state_dict()
    
    save_path = f"/kaggle/working/{model_name}_{percent}.pt"
    torch.save({"model": state, "mean": mean, "std": std}, save_path)
    print("Saved:", save_path)
    
    mae, rmse, r2, mape = evaluate_model(model, val_loader, mean, std)
    return mae, rmse, r2, mape

In [15]:
import gc

def clear_memory():
    gc.collect()
    torch.cuda.empty_cache()

# ↓↓↓ ONLY CHANGE THIS ↓↓↓
RUN_PERCENT = 40
# ↑↑↑ ONLY CHANGE THIS ↑↑↑

results = {}

print(f"\n{'='*10} {RUN_PERCENT}% Missing {'='*10}")
results[RUN_PERCENT] = train_single_dataset(RUN_PERCENT, model_name="pemsbay_iconvlstm")
mae, rmse, r2, mape = results[RUN_PERCENT]

print(f"\nFINAL RESULT {RUN_PERCENT}%")
print(f"MAE  : {mae:.4f}")
print(f"RMSE : {rmse:.4f}")
print(f"R2   : {r2:.4f}")
print(f"MAPE : {mape:.2f}%")

clear_memory()


========== 40% Missing ==========

STARTING EXPERIMENT: 40% MISSING


/usr/local/lib/python3.12/dist-packages/numpy/lib/_nanfunctions_impl.py:1233: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a, func=_nanmedian, keepdims=keepdims,


--- Statistics for 40% Missing ---
Mean (first 5 sensors): [ 0.       67.574814 58.968327 59.07396  62.23572 ]
Std  (first 5 sensors): [ 1.        8.210684 13.359451 11.839966  8.56641 ]
Using 2 GPUs with DataParallel
Parameters: 157,890


Epoch 1/80:   0%|          | 0/652 [00:00<?, ?it/s]

Epoch 01 | Train: 0.3713 | Val: 0.3524 | LR: 1.00e-03
  ✓ Saved checkpoint to /kaggle/working/best_40.pt


...

Epoch 78 | Train: 0.3560 | Val: 0.3461 | LR: 3.13e-05


Epoch 79/80:   0%|          | 0/652 [00:00<?, ?it/s]

Epoch 79 | Train: 0.3560 | Val: 0.3461 | LR: 3.13e-05


Epoch 80/80:   0%|          | 0/652 [00:00<?, ?it/s]

Epoch 80 | Train: 0.3561 | Val: 0.3460 | LR: 3.13e-05
  ✓ Saved checkpoint to /kaggle/working/best_40.pt
Saved: /kaggle/working/pemsbay_iconvlstm_40.pt

Normalized Results
MAE  : 0.3132
RMSE : 0.5746

Denormalized Results (Real Scale)
MAE  : 2.5985
RMSE : 5.0974
R2   : 0.4818
MAPE : 4.82%

FINAL RESULT 40%
MAE  : 2.5985
RMSE : 5.0974
R2   : 0.4818
MAPE : 4.82%
